# 无权重双 T4 NCCL 探针

这个笔记本只检查两张 GPU 是否各自映射到一个 rank，并完成一次 NCCL allreduce。它不下载 MiniMax-H3 权重，也不生成视频。

CPU、mock、builtin-cpu 的结果不能当作通过。通过条件只有：`collective_ok` 为 true，`distinct_gpu_mapping` 为 true，两个 `device_uuid` 不同，`weights_loaded` 为 false。

先做故意失败的日志验收，把 zip 下载到父机并核对 sha256，再跑真正的探针。容器里生成的 zip 还没下载出来，就不算保全。不要在这里停止 Kaggle session。


In [ ]:
import hashlib, json, os, sys
from pathlib import Path

def find_root():
    seeds = [Path.cwd(), Path("/kaggle/working"), Path("/kaggle/input")]
    seen = []
    for seed in seeds:
        if not seed.exists():
            continue
        seen.extend([seed, *seed.glob("*"), *seed.glob("*/*")])
    for candidate in seen:
        probe = candidate / "src" / "minimax_h3_t4" / "runtime" / "dual_gpu_probe.py"
        if probe.is_file():
            return candidate
    raise SystemExit("未找到 dual_gpu_probe.py。请把本探针文件夹上传到 /kaggle/working 或 input。")

ROOT = find_root()
SRC = ROOT / "src"
sys.path.insert(0, str(SRC))
os.environ["PYTHONPATH"] = str(SRC) + os.pathsep + os.environ.get("PYTHONPATH", "")
COMMIT = (ROOT / "COMMIT.txt").read_text(encoding="utf-8").strip() if (ROOT / "COMMIT.txt").is_file() else "git checkout of this probe"
print(json.dumps({"root": str(ROOT), "commit": COMMIT, "weights_download": False}, ensure_ascii=False))


In [ ]:
import torch
count = torch.cuda.device_count() if torch.cuda.is_available() else 0
names = []
if count:
    names = [torch.cuda.get_device_properties(i).name for i in range(count)]
print(json.dumps({"torch": torch.__version__, "cuda_device_count": count, "devices": names}, ensure_ascii=False))
if count != 2:
    raise SystemExit("这个探针需要两张可见 CUDA 设备。不要改去下载权重。")


## 故意失败

两个 rank 会等待 30 秒才报告就绪，父进程的就绪超时是 2 秒。预期进程退出码为 2，子进程被杀掉，并写出可核对的 zip。这一步通过后，才允许跑真正的 allreduce。


In [ ]:
import hashlib, json
from pathlib import Path
from minimax_h3_t4.runtime.dual_gpu_probe import run_probe

FAILURE_DIR = Path("/kaggle/working/h3-weightfree-evidence/failure")
FAILURE_DIR.mkdir(parents=True, exist_ok=True)
failed = False
try:
    run_probe(mode="cuda-nccl", evidence_dir=FAILURE_DIR, ready_timeout=2, collective_timeout=2, hold_seconds=(30, 30))
except Exception as exc:
    failed = True
    print(type(exc).__name__, str(exc)[:500])
if not failed:
    raise SystemExit("故意失败没有失败，不能继续。")

def verify_bundle(directory: Path):
    zip_path = directory / "evidence.zip"
    side = (directory / "evidence.zip.sha256").read_text(encoding="utf-8").split()[0]
    digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
    if digest != side:
        raise SystemExit("sha256 不一致")
    smoke = json.loads((directory / "smoke_result.json").read_text(encoding="utf-8"))
    return {"zip": str(zip_path), "bytes": zip_path.stat().st_size, "sha256": digest, "smoke": smoke}

failure_bundle = verify_bundle(FAILURE_DIR)
if failure_bundle["smoke"].get("collective_ok") is not False or failure_bundle["smoke"].get("model_stage_allowed") is not False:
    raise SystemExit(failure_bundle["smoke"])
print(json.dumps({"failure_bundle": {k: failure_bundle[k] for k in ("zip", "bytes", "sha256")}}, ensure_ascii=False))
print("先把 failure 目录里的 evidence.zip 和 evidence.zip.sha256 下载到父机并核对。下载成功之前不要跑下一格，也不要停止 session。")


## 真正的双卡探针

只有上一格已经在父机核对过失败日志之后，才运行这一格。预算是就绪 60 秒、集合通信 90 秒。失败会自动终止子进程并写 zip，不会下载权重。


In [ ]:
NCCL_DIR = Path("/kaggle/working/h3-weightfree-evidence/nccl")
NCCL_DIR.mkdir(parents=True, exist_ok=True)
result = run_probe(mode="cuda-nccl", evidence_dir=NCCL_DIR, ready_timeout=60, collective_timeout=90, hold_seconds=(0, 0))
bundle = verify_bundle(NCCL_DIR)
summary = {key: result.get(key) for key in ("collective_ok", "distinct_gpu_mapping", "device_count", "weights_loaded", "model_stage_allowed", "gpu_mapping", "sum")}
summary["sha256"] = bundle["sha256"]
summary["zip"] = bundle["zip"]
print(json.dumps(summary, ensure_ascii=False, default=str))
mapping = result.get("gpu_mapping") or {}
uuids = [row.get("device_uuid") for row in mapping.get("ranks", [])]
if not (result.get("collective_ok") and result.get("distinct_gpu_mapping") and len(set(uuids)) == 2 and result.get("weights_loaded") is False):
    raise SystemExit("双卡映射或 NCCL allreduce 未通过。停止在这一步，先下载 nccl 证据 zip。")
print("通过条件已满足。把 nccl 目录的 evidence.zip 与 sha256 下载到父机并核对之后，才可以停止 session。")


## 权重

这一步不下载文件。只有环境变量 `H3_T4_LOCAL_WEIGHT` 指向一个已经存在的文件，并且 `H3_T4_LOCAL_WEIGHT_SHA256` 与文件内容一致时，才报告可以复用。否则停止，不要自动重新下载约 40GB。


In [ ]:
import os
from minimax_h3_t4.runtime.weights import inspect_local_weight
path = os.environ.get("H3_T4_LOCAL_WEIGHT", "")
digest = os.environ.get("H3_T4_LOCAL_WEIGHT_SHA256", "")
if not path:
    print(json.dumps({"reusable": False, "reason": "no local weight path configured", "download": False}, ensure_ascii=False))
else:
    print(json.dumps(inspect_local_weight(name="local", path=path, expected_sha256=digest or None), ensure_ascii=False))
